# LLM clients

A **large language model** takes text in and produces text out. Every provider
wraps that in its own SDK, its own request shape and its own way of reporting
what a call cost. Kaval.AI puts one small async interface in front of all of
them, so switching providers is a change of one string.

Three methods do the work:

| Method | What it does |
|--------|--------------|
| `prompt(message)` | one call, one answer |
| `stream_prompt(message)` | the same call, streamed as it is generated |
| `chat_completions(chat_history=...)` | a full multi-message conversation |

Each takes an optional `response_model` — a [Pydantic](https://docs.pydantic.dev/latest/)
model — to get a **validated object** back instead of prose you have to parse.

The clients are standalone: use them directly for a single call, or let a
{doc}`workflow <workflow>` build them for you.

## Setup

A client needs an API key, read from the environment: `OPENAI_API_KEY`,
`GEMINI_API_KEY`, `ANTHROPIC_API_KEY`, or `OLLAMA_HOST` for a local Ollama.
This notebook lives in `notebooks/`, so it loads the project `.env` next door.

Kaval.AI logs every model call through [loguru](https://loguru.readthedocs.io/).
That is useful in an application and noisy in a tutorial, so we quiet it down to
warnings and print the relevant values explicitly.

In [1]:
import sys

import dotenv
from loguru import logger

dotenv.load_dotenv("../.env")

logger.remove()
_ = logger.add(sys.stderr, level="WARNING")

## Your first call

`make_client` builds a client from a `"provider/model"` id and picks up the
matching API key on its own.

Note what `prompt` returns: a plain `str`. Nothing is wrapped, and there is no
response envelope to dig through.

In [2]:
from kavalai import make_client

client = make_client("openai/gpt-5.4-mini")

answer = await client.prompt(
    "What is the capital of Estonia? Answer in one sentence."
)
print(answer)

The capital of Estonia is Tallinn.


### The same code, a different provider

The provider lives entirely in that id string. Swap it and nothing else changes
— which is what makes it cheap to compare models, or to fall back to a second
provider when the first one is having a bad day.

In [3]:
gemini = make_client("gemini/gemini-3.1-flash-lite")

print(await gemini.prompt(
    "What is the capital of Estonia? Answer in one sentence."
))

The capital of Estonia is Tallinn.


## Structured output

Prose is a poor interface for a program. The moment you need a *field* — a
price, a category, a date — you are writing a parser against text that changes
wording between calls.

Pass a `response_model` instead and you get a validated instance of your own
Pydantic type. The model is constrained to that schema, and the result is
checked before you ever see it.

In [4]:
from pydantic import BaseModel, Field


class City(BaseModel):
    name: str
    country: str
    population: int = Field(description="Approximate number of residents.")
    fun_fact: str


city = await client.prompt("Describe Tallinn.", response_model=City)

print(type(city).__name__)
print("country   :", city.country)
print("population:", city.population)
print("fun fact  :", city.fun_fact)

City
country   : Estonia
population: 437000
fun fact  : Tallinn’s Old Town is one of the best-preserved medieval city centers in Europe and a UNESCO World Heritage Site.


`city.population` is an `int`, not the string `"about 460,000"`. That is the
whole point: you declare the shape once, and every field arrives typed.

Structured output is not provider-specific either — the same `City` model works
against Gemini:

In [5]:
city_gemini = await gemini.prompt("Describe Tallinn.", response_model=City)
print(f"{city_gemini.name}, {city_gemini.country} — "
      f"{city_gemini.population:,} residents")

Tallinn, Estonia — 445,000 residents


Nested models and lists work too, so one call can return a whole small
document. Here the fictional **Green Village** gets an itinerary:

In [6]:
class Stop(BaseModel):
    name: str
    why_visit: str


class Itinerary(BaseModel):
    village: str
    stops: list[Stop]
    advice: str


itinerary = await client.prompt(
    "Plan a half-day visit to Green Village, a tiny village of 104 residents "
    "with a 23-metre grain tower, a pond called Lake Miller and one pub, "
    "The Rusty Anchor. Suggest three stops.",
    response_model=Itinerary,
)

for stop in itinerary.stops:
    print(f"- {stop.name}: {stop.why_visit}")
print("\nAdvice:", itinerary.advice)

- The 23-metre Grain Tower: Start with the village’s most distinctive landmark. It’s the easiest way to orient yourself, take in the scale of this tiny place, and get a quick sense of its agricultural history.
- Lake Miller: Spend time by the pond for a quiet break and a scenic pause. It’s the best spot for a relaxed walk, photos, and a glimpse of everyday village life.
- The Rusty Anchor: Finish at the village’s only pub for a half-day visit. It’s ideal for a drink, a simple meal, and a chance to chat with locals in the heart of the village.

Advice: With only 104 residents, Green Village is best enjoyed slowly. Keep the visit unhurried, combine sightseeing with a meal at The Rusty Anchor, and allow time to wander between the three stops.


## Conversations

`prompt` sends a single message and keeps no state — like every LLM API, the
client is **stateless**. To hold a conversation you pass the messages so far as
a `ChatHistory`, and the model sees the whole thread.

Roles carry meaning: `system` sets behaviour, `user` is the person, `assistant`
is what the model said previously.

In [7]:
from kavalai import ChatHistory, ChatMessage

history = ChatHistory(
    messages=[
        ChatMessage(role="system",
                    content="You are a terse village guide. "
                            "One short sentence."),
        ChatMessage(role="user",
                    content="I'm visiting Green Village tomorrow."),
        ChatMessage(role="assistant",
                    content="Bring boots — Lake Miller is muddy."),
        ChatMessage(role="user", content="What did you just tell me to bring?"),
    ]
)

print(await client.chat_completions(chat_history=history))

Boots.


Appending every turn by hand gets tedious. Workflows do it for you: the engine
records each turn and replays it on the next one, which is what gives a chatbot
memory. See {doc}`observability_storage`.

## Streaming responses

A long answer takes seconds to generate. Streaming shows it as it arrives, so
the reader sees the first words immediately instead of watching a spinner.

`stream_prompt` returns a {class}`~kavalai.Streamer` you iterate. Each chunk has
a `type` — `partial` while generating, `complete` at the end.

In [8]:
streamer = await client.stream_prompt(
    "In three sentences, describe autumn in Tallinn's Old Town."
)

chunks = 0
async for chunk in streamer:
    if chunk.type == "partial":
        chunks += 1
    elif chunk.type == "complete":
        print(chunk.value)

print(f"\n[arrived in {chunks} partial chunks]")

Autumn in Tallinn’s Old Town feels crisp and atmospheric, with golden leaves drifting along the cobblestone streets and medieval rooftops softened by cool, slanting light. The narrow lanes, church spires, and ancient walls take on a quiet, slightly misty charm, especially in the early mornings and evenings. Cafés glow warmly behind old stone facades, making the whole district feel both historic and cozy.

[arrived in 84 partial chunks]


By default each `partial` carries the full text accumulated so far, so a UI can
render it directly with no reassembly. For long outputs prefer `stream_delta`,
where each chunk carries only what is new. Structured output streams too — every
partial is still valid JSON. The {doc}`streamer` tutorial covers both.

## Model statistics and observability

Every call reports a {class}`~kavalai.ModelCallStat`: the model, the prompt
and completion token counts, and the wall-clock duration.

**Tokens** are the units a model reads and writes — roughly ¾ of a word each —
and they are what you are billed for. Prompt tokens are what you sent;
completion tokens are what came back.

By default these are logged through loguru. Pass your own
{class}`~kavalai.ModelStatsReceiver` to send them anywhere — a metrics backend,
a database, or a list you inspect later.

In [9]:
from kavalai import ModelCallStat, ModelStatsReceiver, OpenAIClient


class CollectStats(ModelStatsReceiver):
    def __init__(self):
        self.calls: list[ModelCallStat] = []

    def receive_model_stats(self, stats: ModelCallStat):
        self.calls.append(stats)
        print(
            f"{stats.model}: {stats.prompt_tokens} prompt + "
            f"{stats.completion_tokens} completion = "
            f"{stats.total_tokens} tokens "
            f"in {stats.duration_seconds:.2f}s"
        )


collector = CollectStats()
counted = OpenAIClient("gpt-5.4-mini", model_stats_receiver=collector)

await counted.prompt("Name the three streets of Green Village.")
await counted.prompt("How deep is Lake Miller?")

total = sum(c.total_tokens or 0 for c in collector.calls)
print(f"\ntotal so far: {total} tokens "
      f"over {len(collector.calls)} calls")

openai/gpt-5.4-mini: 14 prompt + 29 completion = 43 tokens in 1.03s


openai/gpt-5.4-mini: 12 prompt + 75 completion = 87 tokens in 1.07s

total so far: 130 tokens over 2 calls


Inside a workflow these are aggregated per run as `WorkflowState.token_usage`
and written to the `model_call_stats` table, where the backoffice UI shows them
per call — see {doc}`../guides/observability`. Multiply the token counts by your
provider's published price to turn them into money; the runtime records usage,
not cost.

## Timeouts and retries

Reliability and sampling live on {class}`~kavalai.LlmClientParameters`.

`timeout_seconds` bounds **each attempt** (default 30s). On transient failures —
rate limits, timeouts, dropped connections, 5xx — the client retries with
exponential backoff and jitter, up to 5 attempts. It does **not** retry what you
should fix yourself: authentication failures, 404s and other bad requests raise
immediately.

In [10]:
from kavalai import LlmClientParameters

careful = OpenAIClient(
    "gpt-5.4-mini",
    llm_client_parameters=LlmClientParameters(
        temperature=0.2,      # lower = more deterministic
        timeout_seconds=60,   # per attempt
    ),
)

print(await careful.prompt(
    "In one word: the official flower of Green Village is the "
    "marsh marigold. Name it."
))

Marsh marigold


Sampling parameters (`temperature`, `top_p`, …) default to `None` and are only
sent when you set them, so each provider's own defaults apply otherwise. That is
deliberate: recent Claude models reject sampling parameters outright, so leaving
them unset is what keeps one `LlmClientParameters` usable across providers.

When you **stream**, a second timeout guards the gap *between* chunks:
`stream_timeout_seconds`, defaulting to twice `timeout_seconds` so a stream
survives one timed-out attempt plus its retry backoff. Each retry pushes a
`restart` chunk, telling consumers to discard what they accumulated — the new
attempt re-sends from the beginning.

## Embeddings

An **embedding** turns text into a list of numbers — a vector — positioned so
that similar meanings land near each other. That is what makes semantic search
possible: you compare vectors instead of matching words.

`make_embedding_client` mirrors `make_client`. `fastembed` runs locally with no
API key, which is why this notebook uses it; `openai`, `gemini` and `ollama`
work the same way.

`compute_embeddings` returns `(vectors, stats)`. With `normalize=True` the
vectors have unit length, so cosine similarity reduces to a dot product.

In [11]:
from kavalai import make_embedding_client

embedder = make_embedding_client("fastembed/BAAI/bge-small-en-v1.5")

texts = [
    "Tallinn is the capital of Estonia.",
    "Estonia's capital city is Tallinn.",
    "Green Village has one pub, The Rusty Anchor.",
]
vectors, stats = await embedder.compute_embeddings(texts, normalize=True)

print(f"{len(vectors)} vectors of dimension {len(vectors[0])}")


def similarity(a, b):
    return sum(x * y for x, y in zip(a, b))


print(f"sim(0, 1) = {similarity(vectors[0], vectors[1]):.3f}"
      "  <- same meaning, different words")
print(f"sim(0, 2) = {similarity(vectors[0], vectors[2]):.3f}  <- unrelated")

3 vectors of dimension 384
sim(0, 1) = 0.976  <- same meaning, different words
sim(0, 2) = 0.427  <- unrelated


The first two sentences express the same proposition with different structure and score
0.98; the third shares the word "Village" with nothing else in common and scores
0.43. The vector captures meaning, not spelling — which is why this beats keyword
search for questions phrased in the user's own words. Indexing a corpus this way
and querying it is exactly what {doc}`rag` does for you.

## Provider clients: OpenAI, Gemini, Anthropic and Ollama

| Provider | Class | Key / host | Notes |
|----------|-------|------------|-------|
| `openai/…` | {class}`~kavalai.OpenAIClient` | `OPENAI_API_KEY` | `base_url` for Azure and OpenAI-compatible endpoints |
| `gemini/…` | {class}`~kavalai.GeminiClient` | `GEMINI_API_KEY` | Google Gemini models |
| `anthropic/…` | {class}`~kavalai.AnthropicClient` | `ANTHROPIC_API_KEY` | Claude models; leave `temperature` / `top_p` unset |
| `ollama/…` | {class}`~kavalai.OllamaClient` | `OLLAMA_HOST` (default `http://localhost:11434`) | runs locally, no API key |
| `browser/…` | {class}`~kavalai.BrowserLLMClient` | — | in-browser WebGPU, see {doc}`run_in_browser` |

You can also construct a client directly and pass the key yourself:

```python
from kavalai import OpenAIClient, GeminiClient, AnthropicClient, OllamaClient

openai = OpenAIClient("gpt-5.4-mini", api_key="sk-...")
gemini = GeminiClient("gemini-3.1-flash-lite")   # reads GEMINI_API_KEY
claude = AnthropicClient("claude-sonnet-5")      # reads ANTHROPIC_API_KEY
ollama = OllamaClient("llama3")                  # local, reads OLLAMA_HOST
```

**Inside a workflow** you rarely build a client at all — you name the model and
the engine builds it, per node or as the workflow default via
`llm_model="openai/gpt-5.4-mini"`. Set `KAVALAI_DEFAULT_LLM_MODEL` and you can
omit it entirely.

## Bringing your own provider

The table above is a starting set, not a closed one. `register_llm_provider`
adds a provider of your own under a name of your choosing, and from that point
it is indistinguishable from a built-in: `mycorp/model-x` works in `make_client`,
as a workflow's `llm_model`, in a YAML node and in
`KAVALAI_DEFAULT_LLM_MODEL`.

Most third-party providers — DeepSeek, Groq, Together, Fireworks, OpenRouter —
speak the OpenAI wire format, so there is nothing to implement. Point the
built-in client at their base URL and give the combination a name:

In [ ]:
from kavalai import OpenAIClient, register_llm_provider

# Any endpoint speaking the OpenAI dialect. This one is OpenAI's own,
# because that is the key this notebook has.
register_llm_provider(
    "mycorp", OpenAIClient, base_url="https://api.openai.com/v1"
)

mycorp = make_client("mycorp/gpt-5.4-mini")
print(await mycorp.prompt("Name the capital of Estonia in three words."))


Tallinn is the capital.


Keyword arguments given at registration are bound to the name and handed to the
client on every use, so one class can serve several endpoints:

```python
register_llm_provider(
    "deepseek", OpenAIClient,
    base_url="https://api.deepseek.com",
    api_key=os.environ["DEEPSEEK_API_KEY"],
)
```

A registration can also name a class that has not been imported yet. The dotted
path is resolved on first use, which is how Kaval.AI registers its own clients —
so `import kavalai` costs nothing on a machine where no provider SDK is
installed:

```python
register_llm_provider("mycorp", "mycorp_sdk.client.MyCorpClient")
```

### A provider with its own protocol

When the API is not OpenAI-shaped, implement {class}`~kavalai.BaseLlmClient`.
There is one method to write, and it does three things: send the request, push
text into the value streamer as it arrives, and report what the call cost.

The client below speaks Anthropic's Messages API over plain HTTP. Kaval.AI
already ships {class}`~kavalai.AnthropicClient`, which is what makes this a
runnable example rather than a sketch: execute it, then read the shipped class
to see what a production client adds — real streaming, retries, refusal
handling and cached-token accounting.

In [ ]:
import json
import os
import time

import httpx

from kavalai import BaseLlmClient, ModelCallStat, ensure_user_turn


class MinimalAnthropicClient(BaseLlmClient):
    """A chat client for a provider with its own request format."""

    provider = "mycorp-native"

    def __init__(self, model, llm_client_parameters=None,
                 model_stats_receiver=None, api_key=None):
        super().__init__(llm_client_parameters, model_stats_receiver)
        self.model = model
        self.api_key = api_key or os.environ["ANTHROPIC_API_KEY"]

    async def _run_chat_completions(self, chat_history, response_model,
                                    streamer):
        started = time.perf_counter()
        value_streamer = streamer.get_value_streamer(
            "response", response_model=response_model
        )

        payload = {
            "model": self.model,
            "max_tokens": 1024,
            # prompt() and llm nodes send system-only histories, which this
            # API rejects; ensure_user_turn is the shared fix.
            "messages": [
                {"role": message.role, "content": message.content}
                for message in ensure_user_turn(chat_history.messages)
            ],
        }
        if response_model:
            schema = response_model.model_json_schema()
            schema["additionalProperties"] = False
            payload["output_config"] = {
                "format": {"type": "json_schema", "schema": schema}
            }

        async with httpx.AsyncClient(timeout=60.0) as http:
            response = await http.post(
                "https://api.anthropic.com/v1/messages",
                headers={"x-api-key": self.api_key,
                         "anthropic-version": "2023-06-01"},
                json=payload,
            )
            response.raise_for_status()
            body = response.json()

        text = "".join(block["text"] for block in body["content"]
                       if block["type"] == "text")
        await value_streamer.stream_partial(text)
        await value_streamer.stream_complete()

        usage = body.get("usage", {})
        prompt_tokens = usage.get("input_tokens", 0)
        completion_tokens = usage.get("output_tokens", 0)
        await self._send_model_call_stats(ModelCallStat(
            call_type="llm",
            model=self.stat_model_name(),
            request_data=json.dumps(payload, default=str),
            response_data=text,
            duration_seconds=time.perf_counter() - started,
            prompt_tokens=prompt_tokens,
            completion_tokens=completion_tokens,
            total_tokens=prompt_tokens + completion_tokens,
            response_code=response.status_code,
        ))


register_llm_provider("mycorp-native", MinimalAnthropicClient)

claude = make_client("mycorp-native/claude-sonnet-5")
print(await claude.prompt("Reply with exactly: it works."))


it works.


Reporting a {class}`~kavalai.ModelCallStat` is what puts a custom provider's
token usage in the same place as the built-ins: `WorkflowState.token_usage`, the
`model_call_stats` table and the backoffice, with nothing else needing to be
told the provider exists.

Structured output needs no extra support either. The value streamer validates
whatever comes back against the model, so a custom client gets typed answers on
the same terms as `openai/`:

In [ ]:
class Village(BaseModel):
    name: str
    residents: int


village = await claude.prompt(
    "Green Village has 104 residents. Describe it.", response_model=Village
)
print(repr(village))


Village(name='Green Village', residents=104)


### Embedding providers

`register_embedding_provider` mirrors it. An embedding client implements
`compute_embeddings`, which returns the vectors and a
{class}`~kavalai.ModelCallStat`.

The one below calls no provider at all: it hashes words into a fixed number of
dimensions with scikit-learn, downloading nothing and needing no key. Be clear
about what that buys — the vectors are lexical rather than semantic. They match
shared words, which is useful for exact terms, offline runs and tests, and no
substitute for an embedding model when the question is worded differently from
the answer.

In [ ]:
from sklearn.feature_extraction.text import HashingVectorizer

from kavalai import (
    BaseEmbeddingClient,
    make_embedding_client,
    register_embedding_provider,
)


class LexicalEmbeddingClient(BaseEmbeddingClient):
    """Hashed bag-of-words vectors: no download, no key, no network."""

    def __init__(self, model, dimensions=512):
        super().__init__(model)
        self.vectorizer = HashingVectorizer(
            n_features=dimensions, norm="l2", alternate_sign=False
        )

    async def compute_embeddings(self, texts, normalize=False,
                                 normalizer=None, **kwargs):
        started = time.perf_counter()
        vectors = self.vectorizer.transform(texts).toarray().tolist()
        stats = ModelCallStat(
            call_type="embedding",
            model=f"lexical/{self.model}",
            batch_size=len(texts),
            total_tokens=0,
            duration_seconds=time.perf_counter() - started,
        )
        return vectors, stats


register_embedding_provider("lexical", LexicalEmbeddingClient, dimensions=512)

embedder = make_embedding_client("lexical/word-hash")
vectors, stats = await embedder.compute_embeddings(["a test sentence"])
print(f"{len(vectors[0])} dimensions, batch of {stats.batch_size}")


512 dimensions, batch of 1


The registry is what carries that beyond the client itself. A RAG service builds
its embedding client by name through the same lookup, so a provider registered in
one line reaches indexing, retrieval and every `rag_query` node without either of
them being changed:

In [ ]:
from kavalai.rag import SqliteRagService

FACTS = [
    "Green Village's oldest resident is Agnes Whitlow (born 02.06.1929).",
    "Green Village has 104 residents.",
    "The village pond, Lake Miller, is 1.2 metres deep at its deepest "
    "point.",
    "Green Village's only pub, The Rusty Anchor, has been operating "
    "since 1923.",
]

village_index = SqliteRagService(":memory:", model="lexical/word-hash")
await village_index.index_batch(
    texts=FACTS,
    metadata_list=[{}] * len(FACTS),
    source_ids=[f"fact-{i}" for i in range(len(FACTS))],
)

for hit in await village_index.query("How deep is Lake Miller?", top_k=2):
    print(f"{hit.similarity:.3f}  {hit.content}")


0.516  The village pond, Lake Miller, is 1.2 metres deep at its deepest point.
0.135  Green Village's oldest resident is Agnes Whitlow (born 02.06.1929).


Two rules matter in practice.

**Register before the workflow loads.** Model names are checked when the graph is
parsed, and a `rag_query` node resolves its service then too. An import two lines
too late looks exactly like a typo.

**A workflow names a registration, never a Python path.** Workflow documents are
served over HTTP and edited in the backoffice, so a dotted path in one would turn
"edit a workflow" into "run code on the agent server". Both that and connection
strings are rejected when the workflow loads.

Under `python -m kavalai.server` nobody constructs the engine in Python, so name
the modules that do the registering and let the server import them first:

```bash
KAVALAI_PROVIDER_MODULES=mycorp.providers,mycorp.rag
```

Every dotted registration is resolved directly afterwards, so a mistyped path
stops the container at boot instead of failing the first request that reaches
that node.

Registering a name that already exists raises, as duplicate tool names do;
`replace=True` overrides it and logs a warning, because it changes what every
later lookup means. `register_rag_service` completes the set — see {doc}`rag`.

## Where to next

- {doc}`streamer` — streaming text and structured values in depth.
- {doc}`agents` — give a model tools and let it act.
- {doc}`workflow` — orchestrate several steps into a typed graph.
- {doc}`rag` — answer from your own documents.
- {doc}`../guides/observability` — tracing, stats and the backoffice UI.